# Duke Point Pollution Control Centre

In [1]:
import datetime as dt
import gsw
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.io as sio
import xarray as xr
import PyCO2SYS as pyco2

In [2]:
j=474
i=217

## Temperature (deg C)

In [10]:
temp_2025 = np.array([10,10,10,10,10,10,10,10,10,10,10,10])
temp = temp_2025
temp

array([10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10])

## Flux (m^3/day)

In [5]:
flux__m3d_2025 = np.array([136.0, 157.2, 161.4, 124.8, 119.4, 123.3, 136.9, 132.9, 128.8, 114.8, 133.1, 164.5])
flux_m3d = flux__m3d_2025
flux_m3d

array([136. , 157.2, 161.4, 124.8, 119.4, 123.3, 136.9, 132.9, 128.8,
       114.8, 133.1, 164.5])

In [6]:
mesh_mask_nc = "/ocean/atall/MOAD/grid/mesh_mask_202310b.nc"
ds_mask = xr.open_dataset(mesh_mask_nc)
rho = 1000
seconds_per_day = 86400
cell_area = (ds_mask["e1t"].isel(t=0).values[j, i]* ds_mask["e2t"].isel(t=0).values[j, i])
new_flux = (flux_m3d * rho/ (cell_area * seconds_per_day))
print("New flux:", new_flux)
print("New flux is in kg/m2/S")

New flux: [7.29269992e-06 8.42950314e-06 8.65471887e-06 6.69212463e-06
 6.40256155e-06 6.61169044e-06 7.34096043e-06 7.12646926e-06
 6.90661581e-06 6.15589670e-06 7.13719382e-06 8.82094953e-06]
New flux is in kg/m2/S


## Ammonia (mg/L)

In [7]:
ammonia_2025 = np.array([0.299, 0.747, 7.618, 2.016, 0.418, 0.353, 0.939, 0.0825, 0.471, 0.323, 0.282, 0.674])
ammonia = ammonia_2025
ammonia

array([0.299 , 0.747 , 7.618 , 2.016 , 0.418 , 0.353 , 0.939 , 0.0825,
       0.471 , 0.323 , 0.282 , 0.674 ])

In [8]:
# Convert mg/L NH3 to mmol/m³
molar_mass_NH3 = 17.031  # g/mol
ammonia_mmol_m3 = ammonia * 1000 / molar_mass_NH3
print(ammonia_mmol_m3)

[ 17.55622101  43.86119429 447.30197874 118.37237978  24.54347954
  20.72690975  55.13475427   4.8441078   27.65545182  18.96541601
  16.55804122  39.57489284]


## Alkalinity

In [9]:
dSi = np.zeros(12)

## BOD (mg/L)

In [11]:
bod_2025 = np.array([2.77, 4.09, 4.23, 4.28, 7.35, 10.94, 3.04, 3.76, 2.78, 4.44, 3.22, 3.43])
bod = bod_2025
bod

array([ 2.77,  4.09,  4.23,  4.28,  7.35, 10.94,  3.04,  3.76,  2.78,
        4.44,  3.22,  3.43])

## PON and DON

In [12]:
# Convert BOD from mg O2/L to mmol O2/m3,then estimate biodegradable organic carbon in mmol C/m3
boc_mmol_m3 = bod * 1000 / 32 * 106 / 138
# Assumed molar carbon-to-nitrogen ratio
carbon_to_nitrogen_ratio = 34
# Assumption: DOC = 0.8 * POC
doc_to_poc_ratio = 0.8
# Split biodegradable organic carbon into particulate and dissolved carbon
poc_mmol_m3 = boc_mmol_m3 / (1 + doc_to_poc_ratio)
doc_mmol_m3 = poc_mmol_m3 * doc_to_poc_ratio
# Convert particulate and dissolved carbon to organic nitrogen
PON = poc_mmol_m3 / carbon_to_nitrogen_ratio
DON = doc_mmol_m3 / carbon_to_nitrogen_ratio

print("Estimated PON in mmol N/m3:")
print(PON)

print("\nEstimated DON in mmol N/m3:")
print(DON)

Estimated PON in mmol N/m3:
[1.0864385  1.60416371 1.65907396 1.67868476 2.88278808 4.29084375
 1.19233684 1.47473241 1.09036066 1.74143933 1.26293573 1.3453011 ]

Estimated DON in mmol N/m3:
[0.8691508  1.28333097 1.32725916 1.34294781 2.30623046 3.432675
 0.95386947 1.17978592 0.87228853 1.39315146 1.01034858 1.07624088]


## pH

In [14]:
ph_2025 = np.array([7.18, 7.02, 6.91, 6.97, 7.39, 7.22, 7.11, 7.25, 7.12, 7.08, 7.03, 6.98])
ph = ph_2025
ph

array([7.18, 7.02, 6.91, 6.97, 7.39, 7.22, 7.11, 7.25, 7.12, 7.08, 7.03,
       6.98])

## DIC

In [15]:
dic = np.zeros(12)

## TSS (mg/L)

In [16]:
tss_2025 = np.array([4.00, 5.10, 7.00, 10.15, 8.56, 11.25, 6.04, 13.35, 7.10, 6.20, 3.14, 5.12])
tss = tss_2025
tss

array([ 4.  ,  5.1 ,  7.  , 10.15,  8.56, 11.25,  6.04, 13.35,  7.1 ,
        6.2 ,  3.14,  5.12])

## Turbidity

In [17]:
turbidity_fau = (tss - 7) / 0.89
print("Turbidity in FAU:")
print(turbidity_fau)

Turbidity in FAU:
[-3.37078652 -2.13483146  0.          3.53932584  1.75280899  4.7752809
 -1.07865169  7.13483146  0.11235955 -0.8988764  -4.33707865 -2.11235955]


## Nitrate

In [18]:
nitrate = np.zeros(12)

## Oxygen

In [19]:
oxygen_2025 = np.array([184,184,184,184,184,184,184,184,184,184,184,184])
oxygen = oxygen_2025
oxygen

array([184, 184, 184, 184, 184, 184, 184, 184, 184, 184, 184, 184])

In [20]:
dSi = np.zeros(12)
diatoms = np.zeros(12)
nanoflagellates = np.zeros(12)
z1 = np.zeros(12)
bSi = np.zeros(12)